# User Behaviour Predection 

In [21]:
# step no 1 : packages

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [22]:
# step no 2 : Loading Datasets

orders = pd.read_csv("olist_orders_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [23]:
# step no 3 : Merge datasets

df = pd.merge(orders, reviews, on='order_id')

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,e73b67b67587f7644d5bd1a52deb1b01,5,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,359d03e676b3c069f62cadba8dd3f6e8,5,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,e50934924e227544ba8246aeb3770dd4,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51


In [24]:
# step no 4 : Check null values

df.isnull().sum()

order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                  156
order_delivered_carrier_date      1756
order_delivered_customer_date     2865
order_estimated_delivery_date        0
review_id                            0
review_score                         0
review_comment_title             87656
review_comment_message           58247
review_creation_date                 0
review_answer_timestamp              0
dtype: int64

In [25]:
# drop unnecessary columns

df = df[['order_purchase_timestamp', 'order_delivered_customer_date',
         'order_estimated_delivery_date', 'review_score']]

df.dropna(inplace=True)

In [26]:
# convert date columns

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

# create delivery time feature
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# create delay feature
df['delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days

df.head()

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_score,delivery_days,delay_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,4,8,-8
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,4,13,-6
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,5,9,-18
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,5,13,-13
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,5,2,-10


In [27]:
# step no 6 : Create target column

# bad review = 1 (1,2,3)
# good review = 0 (4,5)

df['target'] = df['review_score'].apply(lambda x: 1 if x <= 3 else 0)

df['target'].value_counts()

target
0    76047
1    20312
Name: count, dtype: int64

In [28]:
# dataset nature

print(f"Good Reviews: {sum(df['target']==0)}")
print(f"Bad Reviews: {sum(df['target']==1)}")

Good Reviews: 76047
Bad Reviews: 20312


In [29]:
# step no 7 : Feature Selection

X = df[['delivery_days', 'delay_days']]
y = df['target']

In [30]:
# step no 8 : Dataset Splitting

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [31]:
# step no 9 : Handle Imbalance

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_res, y_res = smote.fit_resample(X_train, y_train)

print("Before:\n", y_train.value_counts())
print("After:\n", y_res.value_counts())

Before:
 target
0    60837
1    16250
Name: count, dtype: int64
After:
 target
0    60837
1    60837
Name: count, dtype: int64


In [32]:
# step no 10 : Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_res)
X_test_scaled = scaler.transform(X_test)

In [33]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

# train on balanced data
rf_model.fit(X_res, y_res)

rf_pred = rf_model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_acc)
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.7174138646741387
              precision    recall  f1-score   support

           0       0.83      0.80      0.82     15210
           1       0.35      0.40      0.38      4062

    accuracy                           0.72     19272
   macro avg       0.59      0.60      0.60     19272
weighted avg       0.73      0.72      0.72     19272



In [34]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_scaled, y_res)

lr_pred = lr_model.predict(X_test_scaled)

lr_acc = accuracy_score(y_test, lr_pred)

print("Logistic Regression Accuracy:", lr_acc)
print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.6889788293897883
              precision    recall  f1-score   support

           0       0.85      0.74      0.79     15210
           1       0.34      0.50      0.40      4062

    accuracy                           0.69     19272
   macro avg       0.59      0.62      0.60     19272
weighted avg       0.74      0.69      0.71     19272



In [35]:
model_accuracies = {
    "RF": rf_acc,
    "LR": lr_acc
}

model_accuracies

{'RF': 0.7174138646741387, 'LR': 0.6889788293897883}

In [36]:
best_model_name = max(model_accuracies, key=model_accuracies.get)
best_accuracy = model_accuracies[best_model_name]

print(f"\nBest Model: {best_model_name} with accuracy of {best_accuracy:.4f}")


Best Model: RF with accuracy of 0.7174


In [37]:
import joblib

joblib.dump(rf_model, "user_behavior_model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']